# Embedding and retrieval experiments

Compare multilingual embedding models on selected semantic chunks. Device selection uses CPU for model comparison when the GPU has under 4 GB of VRAM, to avoid kernel crashes. Scores include expected-PDF retrieval and answer-keyword matches.


## 1. Setup

In [1]:
import gc,importlib,json,sys
from pathlib import Path
from time import perf_counter
import numpy as np,pandas as pd,torch
from IPython.display import display
cwd=Path.cwd().resolve(); candidates=[cwd,*cwd.parents,cwd/"SheCraft"/"ai-service",cwd/"ai-service"]
PROJECT_ROOT=next((x for x in candidates if (x/"src"/"rag"/"scripts").is_dir() and (x/"rag_documents").is_dir()),None)
if PROJECT_ROOT is None: raise RuntimeError("Could not locate ai-service project root.")
RAG_SRC_DIR=PROJECT_ROOT/"src"/"rag"
if str(RAG_SRC_DIR) not in sys.path: sys.path.insert(0,str(RAG_SRC_DIR))
CUDA_AVAILABLE=torch.cuda.is_available()
GPU_MEMORY_GB=(torch.cuda.get_device_properties(0).total_memory/1024**3) if CUDA_AVAILABLE else 0.0
# Semantic chunking only runs locally; use CPU below 4 GB VRAM.
CHUNKING_DEVICE="cuda" if CUDA_AVAILABLE and GPU_MEMORY_GB>=4.0 else "cpu"
EMBEDDING_DEVICE="cuda" if CUDA_AVAILABLE and GPU_MEMORY_GB>=4.0 else "cpu"
print("GPU memory (GB):",round(GPU_MEMORY_GB,2))
print("Semantic chunking device:",CHUNKING_DEVICE)
print("Local embedding comparison device:",EMBEDDING_DEVICE)
print("Gemini candidates use the API key; local candidates use Sentence Transformers.")


GPU memory (GB): 2.0
Semantic chunking device: cpu
Local embedding comparison device: cpu
Gemini candidates use the API key; local candidates use Sentence Transformers.


## 2. Select documents and candidate models

In [2]:
SUBSET_FILES = None

# Compare the Gemini API candidates alongside the existing local baselines.
MODEL_CONFIGS = {
    "gemini_embedding_001": {
        "enabled": True,
        "provider": "gemini_api",
        "model_name": "gemini-embedding-001",
        "batch_size": 100,
    },
    "gemini_embedding_2": {
        "enabled": True,
        "provider": "gemini_api",
        "model_name": "gemini-embedding-2",
        "batch_size": 1,
    },
    "multilingual_e5_base": {
        "enabled": True,
        "provider": "sentence_transformers",
        "model_name": "intfloat/multilingual-e5-base",
        "batch_size": 8,
    },
    "paraphrase_multilingual_minilm": {
        "enabled": True,
        "provider": "sentence_transformers",
        "model_name": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "batch_size": 16,
    },
    "bge_m3": {
        "enabled": True,
        "provider": "sentence_transformers",
        "model_name": "BAAI/bge-m3",
        "batch_size": 8,
    },
}

enabled_models = {
    key: value for key, value in MODEL_CONFIGS.items() if value["enabled"]
}
print("Evaluation scope: all documents represented by test cases")
print("Answer hits require expected keywords in a relevant top-5 chunk")
print("Embedding models that will run:", list(enabled_models))


Evaluation scope: all documents represented by test cases
Answer hits require expected keywords in a relevant top-5 chunk
Embedding models that will run: ['gemini_embedding_001', 'gemini_embedding_2', 'multilingual_e5_base', 'paraphrase_multilingual_minilm', 'bge_m3']


## 3. Prepare semantic chunks and load evaluation cases

This cell prepares the selected semantic chunks through the shared pipeline, reusing cached PDF text and saving the selected chunks for later notebooks. It then loads the 100-question benchmark with expected PDF names and answer keywords.


In [3]:
import gc,importlib,json,unicodedata
embedding_module=importlib.reload(importlib.import_module("scripts.5_embeddings"))
pipeline_module=importlib.reload(importlib.import_module("scripts.pipeline"))
DOCUMENTS_DIR=PROJECT_ROOT/"rag_documents"
ARTIFACT_ROOT=PROJECT_ROOT/"data"/"rag"/"processed_documents"
EMBEDDING_ROOT=PROJECT_ROOT/"data"/"rag"/"embeddings"
EVALUATION_DIR=PROJECT_ROOT/"data"/"rag"/"evaluations"
CASES_FILE=EVALUATION_DIR/"retrieval_method_benchmark_100.json"
evaluation_cases=json.loads(CASES_FILE.read_text(encoding="utf-8"))
if len(evaluation_cases)!=100: raise ValueError(f"Expected 100 benchmark cases, found {len(evaluation_cases)}")
missing_keywords=[case["id"] for case in evaluation_cases if not case.get("expected_keywords")]
if missing_keywords: raise ValueError(f"Cases missing expected_keywords: {missing_keywords}")

SELECTED_CHUNKING_STRATEGY="semantic"
CHUNKING_OPTIONS={"similarity_threshold":0.65,"max_chunk_size":800,"unit_prefix":"passage: "}
PDF_FILES=sorted(DOCUMENTS_DIR.rglob("*.pdf"))
if not PDF_FILES: raise FileNotFoundError(f"No PDFs found under {DOCUMENTS_DIR}")
pipeline_result=pipeline_module.prepare_documents_for_rag(
    PDF_FILES,recursive=False,artifact_root=ARTIFACT_ROOT,
    chunking_strategy=SELECTED_CHUNKING_STRATEGY,chunking_options=CHUNKING_OPTIONS,
    chunking_model=None,extract_images=False,embedding_device=CHUNKING_DEVICE,embedding_local_files_only=True,
    extraction_options={"use_ocr_fallback":True},force=False,
)
gc.collect()
if CHUNKING_DEVICE=="cuda":torch.cuda.empty_cache()
chunks=pipeline_result["chunks"].drop_duplicates("chunk_id").reset_index(drop=True)
SUBSET_FILES=sorted({name for case in evaluation_cases for name in case["expected_documents"]})
missing_pdfs=set(SUBSET_FILES)-set(chunks["file_name"])
if missing_pdfs:raise ValueError(f"Evaluation PDFs missing from semantic chunks: {sorted(missing_pdfs)}")
subset_chunks=chunks[chunks["file_name"].isin(SUBSET_FILES)].reset_index(drop=True)

# Validate each answer keyword against its expected source PDF text.
source_text_by_name={doc["file_name"]:" ".join(page["final_text"] for page in doc["pages"]) for doc in pipeline_result["extracted_documents"]}
def normalize_keyword_text(value):
    text=unicodedata.normalize("NFKD",str(value).casefold())
    return "".join(character for character in text if character.isalnum())
unverified=[]
for case in evaluation_cases:
    source_text=normalize_keyword_text(" ".join(source_text_by_name.get(name,"") for name in case["expected_documents"]))
    absent=[term for term in case["expected_keywords"] if normalize_keyword_text(term) not in source_text]
    if absent:unverified.append((case["id"],absent))
if unverified:raise ValueError(f"Benchmark keywords not found in labeled source PDFs: {unverified[:10]}")
cache_status=pipeline_result["artifact_status"]
print("Chunking strategy:",SELECTED_CHUNKING_STRATEGY,CHUNKING_OPTIONS)
print("PDFs:",len(PDF_FILES),"semantic chunks:",len(subset_chunks),"evaluation PDFs:",len(SUBSET_FILES))
print("Chunk cache hits:",int(cache_status["chunks_cache_hit"].sum()),"of",len(cache_status))
print("Evaluation cases:",len(evaluation_cases),"keywords verified:",sum(len(c["expected_keywords"]) for c in evaluation_cases))
display(subset_chunks.groupby("file_name").size().rename("semantic_chunks"))


Chunking strategy: semantic {'similarity_threshold': 0.65, 'max_chunk_size': 800, 'unit_prefix': 'passage: '}
PDFs: 39 semantic chunks: 283 evaluation PDFs: 39
Chunk cache hits: 39 of 39
Evaluation cases: 100 keywords verified: 421


file_name
Ring Size Guide.pdf                        20
alexandrite.pdf                             4
amber.pdf                                   4
amethyst.pdf                                4
ametrine.pdf                                3
aquamarine.pdf                              4
birthstone_guide.pdf                       10
citrine.pdf                                 4
diamond.pdf                                 4
emerald.pdf                                 4
engagement_ring_settings_guide.pdf         30
fancy_color_diamond.pdf                     5
garnet.pdf                                  5
gemstone_diamond_pearl_buying_guide.pdf    11
gold_colors_guide.pdf                      25
iolite.pdf                                  4
jade.pdf                                    5
jewelry_chain_guide.pdf                     7
jewelry_settings_bench_guide.pdf           16
kunzite.pdf                                 4
lapis_lazuli.pdf                            4
moonstone.pdf           

## 4. Embed and evaluate each model

In [4]:
EXPERIMENT_EMBEDDING_ROOT=EMBEDDING_ROOT
# Reload evaluation inputs on every run so edited questions are not stale in notebook memory.
evaluation_cases=json.loads(CASES_FILE.read_text(encoding="utf-8"))
if len(evaluation_cases)!=100: raise ValueError(f"Expected 100 benchmark cases, found {len(evaluation_cases)}")
arabic_case=next(case for case in evaluation_cases if case["id"]=="retrieval_070")
if "Ù" in arabic_case["query"] or "Ø" in arabic_case["query"]: raise ValueError("retrieval_070 is still mojibake; save/reload retrieval_method_benchmark_100.json before running this cell.")
print("Loaded retrieval_070 Arabic query:",arabic_case["query"])
GEMINI_CLIENT=embedding_module.create_gemini_client(PROJECT_ROOT/".env")
TOP_K=5
model_metric_rows=[];case_result_rows=[]
for model_key,config in enabled_models.items():
    model_name=config["model_name"];batch_size=config["batch_size"]
    provider=config["provider"]
    model=None
    query_result={"cache_hit":False}
    if provider=="gemini_api":
        embedding_result=embedding_module.get_or_create_gemini_embeddings(
            subset_chunks["text"].astype(str).tolist(),client=GEMINI_CLIENT,
            text_type="passage",model_name=model_name,
            cache_root=EXPERIMENT_EMBEDDING_ROOT,batch_size=batch_size,
        )
        chunk_embeddings=embedding_result["embeddings"]
        chunk_embedding_seconds=embedding_result["elapsed_seconds"]
        queries=[case["query"] for case in evaluation_cases]
        query_result=embedding_module.get_or_create_gemini_embeddings(
            queries,client=GEMINI_CLIENT,text_type="query",model_name=model_name,
            cache_root=EXPERIMENT_EMBEDDING_ROOT,batch_size=batch_size,
        )
        query_embeddings=query_result["embeddings"]
        query_embedding_seconds=query_result["elapsed_seconds"]
    else:
        model_kwargs={"torch_dtype":torch.float16} if EMBEDDING_DEVICE=="cuda" else None
        if EMBEDDING_DEVICE=="cuda":torch.cuda.empty_cache()
        model=embedding_module.load_embedding_model(
            model_name,device=EMBEDDING_DEVICE,local_files_only=True,model_kwargs=model_kwargs
        )
        started=perf_counter()
        embedding_result=embedding_module.get_or_create_chunk_embeddings(
            subset_chunks,model_name=model_name,embedding_root=EXPERIMENT_EMBEDDING_ROOT,
            model=model,batch_size=batch_size,
        )
        chunk_embedding_seconds=perf_counter()-started
        chunk_embeddings=embedding_result["embeddings"]
        queries=[case["query"] for case in evaluation_cases]
        started=perf_counter()
        query_embeddings=embedding_module.embed_queries(
            queries,model=model,model_name=model_name,batch_size=batch_size
        )
        query_embedding_seconds=perf_counter()-started
    scores,positions=embedding_module.cosine_search(query_embeddings,chunk_embeddings,top_k=min(TOP_K,len(subset_chunks)))
    document_ranks=[];answer_ranks=[];document_coverage=[];keyword_coverage=[]
    document_hits={1:[],3:[],5:[]};answer_hits={1:[],3:[],5:[]}
    for case_index,case in enumerate(evaluation_cases):
        expected_documents=set(case["expected_documents"])
        expected_keywords=list(case["expected_keywords"])
        normalized_keywords={term:normalize_keyword_text(term) for term in expected_keywords}
        found_documents=set();found_keywords=set();first_document_rank=None;first_answer_rank=None;top_results=[]
        for rank,position in enumerate(positions[case_index],start=1):
            chunk=subset_chunks.iloc[int(position)]
            document_relevant=chunk["file_name"] in expected_documents
            normalized_chunk=normalize_keyword_text(chunk["text"])
            matched=[term for term,normalized in normalized_keywords.items() if normalized in normalized_chunk]
            answer_relevant=document_relevant and bool(matched)
            if document_relevant:
                found_documents.add(chunk["file_name"])
                if first_document_rank is None:first_document_rank=rank
            if answer_relevant:
                found_keywords.update(matched)
                if first_answer_rank is None:first_answer_rank=rank
            top_results.append({
                "rank":rank,"chunk_id":chunk["chunk_id"],"file_name":chunk["file_name"],
                "page_number":int(chunk["page_number"]),"score":round(float(scores[case_index,rank-1]),6),
                "document_relevant":bool(document_relevant),"answer_relevant":bool(answer_relevant),
                "matched_keywords":matched,
            })
        document_ranks.append(first_document_rank);answer_ranks.append(first_answer_rank)
        for k in document_hits:document_hits[k].append(first_document_rank is not None and first_document_rank<=k)
        for k in answer_hits:answer_hits[k].append(first_answer_rank is not None and first_answer_rank<=k)
        document_coverage.append(len(found_documents)/len(expected_documents) if expected_documents else 0.0)
        keyword_coverage.append(len(found_keywords)/len(expected_keywords) if expected_keywords else 0.0)
        case_result_rows.append({
            "model_key":model_key,"model_name":model_name,"case_id":case["id"],"query":case["query"],
            "language":case["language"],"cross_pdf":case["cross_pdf"],
            "expected_documents":case["expected_documents"],"expected_keywords":expected_keywords,
            "first_document_rank":first_document_rank,"first_answer_rank":first_answer_rank,
            "target_document_coverage_at_5":document_coverage[-1],"keyword_coverage_at_5":keyword_coverage[-1],
            "matched_keywords_at_5":sorted(found_keywords),"top_5":top_results,
        })
    model_metric_rows.append({
        "model_key":model_key,"model_name":model_name,"dimension":int(chunk_embeddings.shape[1]),
        "chunks":len(subset_chunks),"cases":len(evaluation_cases),
        "document_recall_at_1":float(np.mean(document_hits[1])),
        "document_recall_at_3":float(np.mean(document_hits[3])),
        "document_recall_at_5":float(np.mean(document_hits[5])),
        "document_mrr_at_5":float(np.mean([0 if rank is None else 1/rank for rank in document_ranks])),
        "target_document_coverage_at_5":float(np.mean(document_coverage)),
        "answer_recall_at_1":float(np.mean(answer_hits[1])),
        "answer_recall_at_3":float(np.mean(answer_hits[3])),
        "answer_recall_at_5":float(np.mean(answer_hits[5])),
        "answer_mrr_at_5":float(np.mean([0 if rank is None else 1/rank for rank in answer_ranks])),
        "keyword_coverage_at_5":float(np.mean(keyword_coverage)),
        "chunk_embedding_seconds":round(chunk_embedding_seconds,3),
        "chunk_cache_hit":bool(embedding_result["cache_hit"]),
        "query_cache_hit":bool(query_result["cache_hit"]),
        "query_embedding_seconds":round(query_embedding_seconds,3),
        "cache_hit":bool(embedding_result["cache_hit"]),
        "embedding_file_bytes":embedding_result["embeddings_file"].stat().st_size,
    })
    del model,chunk_embeddings,query_embeddings,embedding_result,query_result;gc.collect()
    if EMBEDDING_DEVICE=="cuda":torch.cuda.empty_cache()
model_metrics=pd.DataFrame(model_metric_rows).set_index("model_key")
case_results=pd.DataFrame(case_result_rows)
display(model_metrics.sort_values(["answer_recall_at_5","keyword_coverage_at_5","answer_mrr_at_5"],ascending=False))


Loaded retrieval_070 Arabic query: كيف يصف الدليل حجر العقيق من حيث اللون والتركيب والمصدر؟


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

,model_name,dimension,chunks,cases,document_recall_at_1,document_recall_at_3,document_recall_at_5,document_mrr_at_5,target_document_coverage_at_5,answer_recall_at_1,answer_recall_at_3,answer_recall_at_5,answer_mrr_at_5,keyword_coverage_at_5,chunk_embedding_seconds,chunk_cache_hit,query_cache_hit,query_embedding_seconds,cache_hit,embedding_file_bytes
model_key,,,,,,,,,,,,,,,,,,,,
gemini_embedding_2,gemini-embedding-2,3072,283,100,0.95,0.99,0.99,0.966667,0.970000,0.85,0.98,0.98,0.910000,0.92850,0.000,True,True,0.000,True,3477632
gemini_embedding_001,gemini-embedding-001,3072,283,100,0.92,0.99,1.00,0.952500,0.990000,0.86,0.96,0.98,0.910000,0.91400,0.000,True,True,0.000,True,3477632
bge_m3,BAAI/bge-m3,1024,283,100,0.87,0.89,0.93,0.885167,0.920000,0.79,0.86,0.93,0.836667,0.85000,0.073,True,False,15.221,True,1159296
multilingual_e5_base,intfloat/multilingual-e5-base,768,283,100,0.79,0.89,0.89,0.830000,0.876667,0.72,0.87,0.88,0.783667,0.77025,0.030,True,False,5.051,True,869504
paraphrase_multilingual_minilm,sentence-transformers/paraphrase-multilingual-...,384,283,100,0.68,0.78,0.82,0.730667,0.798333,0.59,0.76,0.80,0.674000,0.68400,0.022,True,False,1.257,True,434816


## 5. Inspect failures and model disagreements

In [5]:
failures=case_results[case_results["first_answer_rank"].isna()]
print("Model/query runs without a keyword-matched answer chunk:",len(failures))
display(failures[["model_key","case_id","query","expected_documents","expected_keywords","language"]])
rank_comparison=case_results.pivot(index=["case_id","query"],columns="model_key",values="first_answer_rank").reset_index()
display(rank_comparison)

# Break down answer-bearing retrieval and keyword coverage by language and cross-PDF status.
evaluation_breakdown=(
    case_results.groupby(["model_key","language","cross_pdf"],dropna=False)
    .agg(
        cases=("case_id","count"),
        answer_recall_at_5=("first_answer_rank",lambda ranks:ranks.notna().mean()),
        keyword_coverage_at_5=("keyword_coverage_at_5","mean"),
        target_document_coverage_at_5=("target_document_coverage_at_5","mean"),
    )
    .reset_index()
)
display(evaluation_breakdown)

Model/query runs without a keyword-matched answer chunk: 43


,model_key,case_id,query,expected_documents,expected_keywords,language
46,gemini_embedding_001,retrieval_047,How can chain-link styles be combined when cre...,"[jewelry_chain_guide.pdf, necklace_stacking_gu...","[link styles, box chains, cable link, layered ...",en
69,gemini_embedding_001,retrieval_070,كيف يصف الدليل حجر العقيق من حيث اللون والتركي...,[garnet.pdf],"[closely related minerals, almost every color,...",ar
146,gemini_embedding_2,retrieval_047,How can chain-link styles be combined when cre...,"[jewelry_chain_guide.pdf, necklace_stacking_gu...","[link styles, box chains, cable link, layered ...",en
169,gemini_embedding_2,retrieval_070,كيف يصف الدليل حجر العقيق من حيث اللون والتركي...,[garnet.pdf],"[closely related minerals, almost every color,...",ar
246,multilingual_e5_base,retrieval_047,How can chain-link styles be combined when cre...,"[jewelry_chain_guide.pdf, necklace_stacking_gu...","[link styles, box chains, cable link, layered ...",en
261,multilingual_e5_base,retrieval_062,كيف يصف الدليل حجر الكهرمان من حيث اللون والتر...,[amber.pdf],"[fossilized tree resin, animal and plant inclu...",ar
262,multilingual_e5_base,retrieval_063,ما الذي يميّز حجر الجمشت ويجعل لونه أو تأثيره ...,[amethyst.pdf],"[amethyst, quartz, purple, ruby, emerald]",ar
265,multilingual_e5_base,retrieval_066,كيف يصف الدليل حجر السترين من حيث اللون والترك...,[citrine.pdf],"[citrine, quartz, yellow, topaz, amethyst]",ar
266,multilingual_e5_base,retrieval_067,ما الذي يميّز حجر الألماس ويجعل لونه أو تأثيره...,[diamond.pdf],"[diamond, carbon, hardness, Mohs]",ar
268,multilingual_e5_base,retrieval_069,ما الخصائص المميزة لحجر الألماس الملون المذكور...,[fancy_color_diamond.pdf],"[fancy color diamonds, strength and purity, ra...",ar


model_key,case_id,query,bge_m3,gemini_embedding_001,gemini_embedding_2,multilingual_e5_base,paraphrase_multilingual_minilm
0,retrieval_001,Should pearl jewelry be cleaned with a steam c...,1.0,1.0,1.0,1.0,1.0
1,retrieval_002,"Why do pearls scratch easily, and what routine...",1.0,1.0,1.0,1.0,3.0
2,retrieval_003,How should a jeweler measure a finger for an a...,1.0,1.0,1.0,1.0,1.0
3,retrieval_004,How can I find a ring size that fits snugly bu...,1.0,1.0,2.0,1.0,1.0
4,retrieval_005,Which ring metals are best for everyday wear a...,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...
95,retrieval_096,Comment nettoyer les perles sans les rayer ni ...,1.0,1.0,1.0,1.0,1.0
96,retrieval_097,"Quels metaux de bague allient resistance, entr...",1.0,1.0,1.0,NaN,1.0
97,retrieval_098,Quels conseils aident ? choisir une taille de ...,1.0,1.0,2.0,1.0,1.0
98,retrieval_099,Quels outils fiables mesurent une taille de ba...,2.0,1.0,2.0,1.0,1.0


,model_key,language,cross_pdf,cases,answer_recall_at_5,keyword_coverage_at_5,target_document_coverage_at_5
0,bge_m3,ar,False,20,0.700000,0.597500,0.700000
1,bge_m3,en,False,41,0.975610,0.939024,0.975610
2,bge_m3,en,True,19,1.000000,0.931579,0.947368
3,bge_m3,fr,False,20,1.000000,0.842500,1.000000
4,gemini_embedding_001,ar,False,20,0.950000,0.870000,1.000000
5,gemini_embedding_001,en,False,41,1.000000,0.969512,1.000000
6,gemini_embedding_001,en,True,19,0.947368,0.865789,0.947368
7,gemini_embedding_001,fr,False,20,1.000000,0.890000,1.000000
8,gemini_embedding_2,ar,False,20,0.950000,0.950000,0.950000
9,gemini_embedding_2,en,False,41,1.000000,0.969512,1.000000


## 6. Save the evaluation and select the best model

In [6]:
ranked_models=model_metrics.reset_index().sort_values(
    ["answer_recall_at_5","keyword_coverage_at_5","answer_mrr_at_5"],ascending=False
).reset_index(drop=True)
RECOMMENDED_MODEL_KEY=ranked_models.iloc[0]["model_key"]
RECOMMENDED_MODEL_NAME=ranked_models.iloc[0]["model_name"]
RESULTS_FILE=EVALUATION_DIR/"embedding_model_comparison_results.json"
RESULTS_FILE.parent.mkdir(parents=True,exist_ok=True)
payload={
    "schema_version":2,
    "chunking_strategy":SELECTED_CHUNKING_STRATEGY,
    "chunking_options":CHUNKING_OPTIONS,
    "chunking_device":CHUNKING_DEVICE,
    "embedding_device":EMBEDDING_DEVICE,
    "subset_files":SUBSET_FILES,
    "model_configs":enabled_models,
    "recommended_model_key":RECOMMENDED_MODEL_KEY,
    "recommended_model_name":RECOMMENDED_MODEL_NAME,
    "selection_rule":["answer_recall_at_5","keyword_coverage_at_5","answer_mrr_at_5"],
    "evaluation_case_file":str(CASES_FILE.relative_to(PROJECT_ROOT)),
    "metrics":model_metrics.reset_index().to_dict(orient="records"),
    "case_results":case_result_rows,
}
RESULTS_FILE.write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding="utf-8")
print("RECOMMENDED MODEL:",RECOMMENDED_MODEL_NAME,"\nResults:",RESULTS_FILE)
display(ranked_models)


RECOMMENDED MODEL: gemini-embedding-2 
Results: C:\Users\User\Desktop\LLMs - Samer Saab\Project\Jewelry\SheCraft\ai-service\data\rag\evaluations\embedding_model_comparison_results.json


,model_key,model_name,dimension,chunks,cases,document_recall_at_1,document_recall_at_3,document_recall_at_5,document_mrr_at_5,target_document_coverage_at_5,...,answer_recall_at_3,answer_recall_at_5,answer_mrr_at_5,keyword_coverage_at_5,chunk_embedding_seconds,chunk_cache_hit,query_cache_hit,query_embedding_seconds,cache_hit,embedding_file_bytes
0,gemini_embedding_2,gemini-embedding-2,3072,283,100,0.95,0.99,0.99,0.966667,0.970000,...,0.98,0.98,0.910000,0.92850,0.000,True,True,0.000,True,3477632
1,gemini_embedding_001,gemini-embedding-001,3072,283,100,0.92,0.99,1.00,0.952500,0.990000,...,0.96,0.98,0.910000,0.91400,0.000,True,True,0.000,True,3477632
2,bge_m3,BAAI/bge-m3,1024,283,100,0.87,0.89,0.93,0.885167,0.920000,...,0.86,0.93,0.836667,0.85000,0.073,True,False,15.221,True,1159296
3,multilingual_e5_base,intfloat/multilingual-e5-base,768,283,100,0.79,0.89,0.89,0.830000,0.876667,...,0.87,0.88,0.783667,0.77025,0.030,True,False,5.051,True,869504
4,paraphrase_multilingual_minilm,sentence-transformers/paraphrase-multilingual-...,384,283,100,0.68,0.78,0.82,0.730667,0.798333,...,0.76,0.80,0.674000,0.68400,0.022,True,False,1.257,True,434816


## 7. Release GPU memory

In [7]:
for variable_name in ["model","chunk_embeddings","query_embeddings","embedding_result","query_result"]:
    globals().pop(variable_name,None)
gc.collect()
if EMBEDDING_DEVICE=="cuda":
    torch.cuda.empty_cache(); torch.cuda.synchronize()
    print("GPU allocated after cleanup (MB):",round(torch.cuda.memory_allocated()/1024**2,2))
print("Embedding comparison complete and model memory released.")


Embedding comparison complete and model memory released.
